# What an oil price shock does to the US economy

A small macroeconomic model, run with Dynare. Every function here is explained in
`doc/lisp_macro_models_manual.md`, which also says what the model leaves out.
Responses are by the quarter: quarter 1 is the quarter the price of oil rises.

## 1. The model with its parameters chosen by hand

`oil-model` runs Dynare (a second or two). `oil-shock-responses` is a table of what happens when the
real price of oil rises 10%; GDP, consumption and the real wage are percent from normal, inflation and
the Fed's rate are points a year.

In [ ]:
(define base (oil-model))
(display-table (oil-shock-responses base :rise 10))

In [ ]:
(plot-oil-shock base :rise 10)

## 2. Scenarios: change a parameter, compare

`wage_rigidity` is how rigid real wages are (near 1: workers resist a cut in what their pay buys, as in the 1970s).
`core_weight` is how much the Fed looks at core inflation rather than headline. Ten quarters this time.

In [ ]:
(define rigid (oil-model :parameters '((wage_rigidity . 0.9)) :quarters 10))
(define flexible (oil-model :parameters '((wage_rigidity . 0.1)) :quarters 10))
(plot-oil-shock (list (list "wages rigid" rigid) (list "wages flexible" flexible)) :rise 10)

In [ ]:
(define looks-through (oil-model :parameters '((core_weight . 1))))
(define reacts (oil-model :parameters '((core_weight . 0))))
(plot-oil-shock (list (list "Fed looks through oil" looks-through) (list "Fed reacts to oil" reacts)) :rise 20)

## 3. The parameters estimated from US data, 1985 to 2008

This takes about five minutes. `:draws 2000` gives a quicker first look.

In [ ]:
(define recent (oil-model-estimate creds))
(show-oil-estimates recent)

### What the estimate is made of

The estimation produces *draws*: each row is one complete set of the 18 estimated numbers (the parameters, the sizes of the shocks and
the measurement error), a small random step from the row before. The estimate of a number is the mean of its column, and its 90%
interval holds 90% of the column. The manual's "What the estimation produces" explains the steps.

In [ ]:
(define draws (hash-table-ref recent "draws"))
(display-table (table-head draws 6))
(vector-mean (table-column draws "price_stickiness"))   ; the estimate of price_stickiness

In [ ]:
(plot-oil-shock recent :rise 10)

## 4. A scenario that starts from the estimate

In [ ]:
(define headline-fed (oil-model :from recent :parameters '((core_weight . 0))))
(plot-oil-shock (list (list "as estimated" recent) (list "Fed watches headline inflation" headline-fed)) :rise 10)

## 5. The 1970s and the 2000s

The same model estimated on 1966 to 1984 (another five minutes), and the 1966-1984 estimate again with
flexible real wages.

In [ ]:
(define early (oil-model-estimate creds :start (date 1966 1 1) :end (date 1984 10 1)))
(show-oil-estimates early)
(plot-oil-shock (list (list "1966-1984" early) (list "1985-2008" recent)) :rise 10)

In [ ]:
(define early-flexible (oil-model :from early :parameters '((wage_rigidity . 0.2))))
(plot-oil-shock (list (list "1966-1984" early) (list "1966-1984, flexible wages" early-flexible)) :rise 10)